# Data Drift

In this section, we will focus on the data drift.
We will check following;
- Drift in raw X features
- Drift in y features
- Drift in preidction => chekcing eval metrics

**Drift in raw features**

We will use Wasserstein, because Kolmogorov Smirnov is too sensive when the data sample is too big 

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from evidently import Dataset, DataDefinition, Report
from evidently.presets import DataDriftPreset
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,mean_absolute_percentage_error
import pathlib
import joblib
from pathlib import Path
import sys
sys.path.append('../src')
from py_def_class import Add_Column,RFPermutationRegressorSelector,check_data_drift,make_drift_table,get_drift_summary,get_drift_share,historical_drift_backtest,historical_drift_backtest_transformed,historical_drift_backtest_target
pd.set_option('display.max_columns', None)

In [8]:
X_old = pd.read_csv(r'../data/data_drift/X_features_old.csv')
X_new = pd.read_csv(r'../data/data_drift/X_new.csv')

In [9]:
binary_cols = [
    b for b in X_old.select_dtypes(exclude='object').columns
    if set(X_old[b].dropna().unique()).issubset({0, 1})
]

drop_feat = binary_cols + ['gp_id','Season','race_round']

X_old_numeric = X_old.drop(columns=drop_feat,axis=1).select_dtypes(exclude='object')
X_old_binary = X_old[binary_cols]
X_old_cat = X_old.select_dtypes(include='object')

X_new_numeric = X_new.drop(columns=drop_feat,axis=1).select_dtypes(exclude='object')
X_new_binary = X_new[binary_cols]
X_new_cat = X_new.select_dtypes(include='object')

X_old_numeric.describe()

,AirTemp_p1,Humidity_p1,Pressure_p1,laptime_sum_sectortimes_p1,LapTimeDiff_p1,AirTemp_p2,Humidity_p2,Pressure_p2,laptime_sum_sectortimes_p2,LapTimeDiff_p2,AirTemp_p3,Humidity_p3,Pressure_p3,laptime_sum_sectortimes_p3,LapTimeDiff_p3,AirTemp_sprint_quali,Humidity_sprint_quali,Pressure_sprint_quali,laptime_sum_sectortimes_sprint_quali,LapTimeDiff_sprint_quali,AirTemp_quali,Humidity_quali,Pressure_quali,Circut_length,Turns,Flat_out_run,Slow_turns,Medium_turns,High_speed_turns,Turn_density
count,3152.000000,3152.000000,3152.000000,3152.000000,3152.000000,2945.000000,2945.000000,2945.000000,2945.000000,2945.000000,2790.000000,2790.000000,2790.000000,2790.000000,2790.000000,238.00000,238.000000,238.000000,238.000000,238.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000
mean,23.836199,52.283027,989.032202,86.758032,1.776235,24.022309,50.537351,986.672224,85.247077,1.384696,23.346487,55.710287,988.130717,86.116293,1.523863,25.02563,53.529412,980.063445,89.424361,1.614000,23.451335,55.832611,986.398487,5.175083,16.374184,1100.642433,5.050742,6.080415,5.243027,3.214401
std,5.270984,16.833499,45.625341,12.729533,3.883422,5.005051,16.157192,51.805110,11.388107,2.327078,5.267680,16.780251,51.399677,12.459379,2.397282,5.82602,22.361745,36.179666,16.026108,5.353138,5.104716,18.300266,49.998752,0.817172,3.339020,321.886508,2.822982,1.268287,3.189846,0.731888
min,11.200000,11.000000,781.700000,54.546000,0.000000,10.900000,19.900000,779.800000,54.506000,0.000000,11.300000,11.400000,782.700000,54.064000,0.000000,16.60000,12.000000,920.100000,64.440000,0.000000,11.500000,12.000000,780.000000,3.340000,10.000000,670.000000,2.000000,3.000000,0.000000,1.900000
25%,19.600000,40.500000,985.600000,77.398750,0.626000,20.200000,37.300000,985.200000,76.835000,0.603000,19.500000,43.000000,988.000000,77.066250,0.665750,21.50000,38.000000,961.550000,80.063000,0.478000,20.000000,42.000000,983.500000,4.380000,14.000000,900.000000,3.000000,5.000000,3.000000,2.860000
50%,24.000000,51.000000,1005.800000,87.274000,1.237000,24.200000,50.800000,1005.000000,84.373000,1.125000,24.100000,56.200000,1005.900000,85.577500,1.194500,24.60000,51.500000,994.000000,90.858000,0.805000,24.100000,57.000000,1005.500000,5.280000,16.000000,1050.000000,4.000000,6.000000,4.000000,3.060000
75%,28.000000,64.800000,1014.400000,95.041500,1.981000,27.800000,61.900000,1013.700000,93.139000,1.698000,27.300000,66.000000,1014.000000,93.887250,1.803750,28.47500,71.000000,1014.575000,101.092750,1.270500,27.000000,69.000000,1013.900000,5.810000,19.000000,1200.000000,5.000000,7.000000,7.000000,3.495000
max,36.200000,94.300000,1026.700000,189.423000,98.363000,37.900000,95.000000,1025.900000,150.472000,55.747000,37.600000,96.000000,1025.400000,157.093000,59.582000,35.70000,94.000000,1018.100000,177.364000,62.164000,35.600000,95.000000,1024.100000,7.000000,27.000000,2200.000000,13.000000,8.000000,17.000000,5.690000


In [10]:
X_new_numeric.describe()

,AirTemp_p1,Humidity_p1,Pressure_p1,laptime_sum_sectortimes_p1,LapTimeDiff_p1,AirTemp_p2,Humidity_p2,Pressure_p2,laptime_sum_sectortimes_p2,LapTimeDiff_p2,AirTemp_p3,Humidity_p3,Pressure_p3,laptime_sum_sectortimes_p3,LapTimeDiff_p3,AirTemp_sprint_quali,Humidity_sprint_quali,Pressure_sprint_quali,laptime_sum_sectortimes_sprint_quali,LapTimeDiff_sprint_quali,AirTemp_quali,Humidity_quali,Pressure_quali,Circut_length,Turns,Flat_out_run,Slow_turns,Medium_turns,High_speed_turns,Turn_density
count,217.000000,217.000000,217.000000,217.000000,217.000000,148.000000,148.000000,148.000000,148.000000,148.000000,150.000000,150.000000,150.000000,150.000000,150.000000,85.000000,85.000000,85.000000,85.000000,85.000000,239.000000,239.000000,239.000000,239.000000,239.000000,239.000000,239.000000,239.000000,239.000000,239.000000
mean,22.710138,42.476498,1003.337327,86.309332,1.836899,24.693243,42.172297,990.236486,83.389054,1.869061,24.270667,44.712667,992.122667,82.835713,1.653313,23.075294,35.602353,1017.827059,87.586682,1.811071,24.676569,39.536820,1000.855649,5.079331,16.184100,1071.439331,5.012552,6.196653,4.974895,3.269331
std,5.670986,13.265245,24.246696,10.909493,1.251713,4.478652,13.664039,25.243713,12.661824,3.280878,5.365920,14.394412,25.722565,12.270361,1.135101,5.371727,11.997261,8.156073,8.274677,3.566592,5.775633,14.232865,24.713789,0.958830,2.878418,240.128142,2.882111,1.269813,3.001295,0.821025
min,12.600000,25.400000,942.700000,67.796000,0.000000,16.600000,26.000000,942.100000,67.014000,0.000000,15.900000,26.900000,944.100000,67.096000,0.000000,15.700000,22.200000,1009.700000,72.965000,0.000000,16.300000,22.700000,942.800000,3.340000,10.000000,670.000000,2.000000,3.000000,0.000000,2.320000
25%,17.200000,30.800000,1008.000000,76.883000,0.898000,22.600000,30.275000,966.200000,74.830000,0.845500,20.800000,28.700000,968.900000,74.830250,0.832500,19.700000,24.400000,1009.900000,88.091000,0.551000,20.500000,25.550000,984.200000,4.360000,14.000000,900.000000,3.000000,6.000000,2.000000,2.940000
50%,23.100000,41.000000,1011.300000,84.620000,1.725000,23.650000,40.200000,1006.200000,80.045000,1.473000,23.100000,52.300000,1007.200000,79.417000,1.553000,24.600000,34.500000,1010.300000,89.707000,1.533000,23.400000,34.500000,1008.700000,5.280000,16.000000,1050.000000,4.000000,6.000000,4.000000,3.060000
75%,24.500000,49.900000,1014.000000,92.803000,2.551000,29.500000,58.600000,1011.500000,91.454750,2.113250,31.000000,57.100000,1014.200000,90.643500,2.360750,30.200000,51.800000,1026.200000,92.161000,2.113000,30.300000,54.350000,1017.450000,5.810000,19.000000,1300.000000,6.000000,7.000000,9.000000,3.210000
max,31.800000,72.400000,1030.000000,112.808000,6.459000,32.100000,66.600000,1011.800000,118.806000,39.077000,32.600000,65.400000,1017.300000,110.631000,5.344000,30.800000,55.900000,1026.600000,121.708000,32.985000,34.100000,66.600000,1028.800000,7.000000,20.000000,1500.000000,13.000000,8.000000,9.000000,5.690000


In [18]:
def check_data_drift(
        X_old,
        X_new,
        drift_share=0.92):

    binary_cols = [
        col for col in X_old.select_dtypes(exclude="object").columns
        if set(X_old[col].dropna().unique()).issubset({0, 1})
    ]

    numeric_cols = [
        col for col in X_old.select_dtypes(exclude="object").columns
        if col not in binary_cols
    ]

    categorical_cols = list(
        X_old.select_dtypes(include="object").columns
    )

    categorical_cols = categorical_cols + binary_cols

    schema = DataDefinition(
        numerical_columns=numeric_cols,
        categorical_columns=categorical_cols
    )

    reference_data = Dataset.from_pandas(
        X_old[numeric_cols + categorical_cols],
        data_definition=schema
    )

    current_data = Dataset.from_pandas(
        X_new[numeric_cols + categorical_cols],
        data_definition=schema
    )

    report = Report([
        DataDriftPreset(
            drift_share=drift_share
        )
    ])

    result = report.run(
        current_data=current_data,
        reference_data=reference_data
    )

    return result.dict()

def make_drift_table(drift_result):

    rows = []

    for metric in drift_result["metrics"]:

        config = metric["config"]

        # We only want individual feature drift metrics
        if config["type"] != "evidently:metric_v2:ValueDrift":
            continue

        feature = config["column"]
        method = config["method"]
        threshold = config["threshold"]
        score = metric["value"]

        # p-value based tests:
        # smaller score = stronger evidence of drift
        if "p_value" in method.lower():
            drift_detected = score <= threshold

        # distance / divergence based methods:
        # larger score = more drift
        else:
            drift_detected = score >= threshold

        rows.append({
            "feature": feature,
            "method": method,
            "score": score,
            "threshold": threshold,
            "drift_detected": drift_detected
        })

    return pd.DataFrame(rows)

def get_drift_summary(drift_result):

    overall = next(
        metric for metric in drift_result["metrics"]
        if metric["config"]["type"] ==
        "evidently:metric_v2:DriftedColumnsCount"
    )

    drifted_count = int(overall["value"]["count"])
    drift_share = overall["value"]["share"]
    drift_threshold = overall["config"]["drift_share"]

    total_features = round(drifted_count / drift_share)

    return pd.DataFrame({
        "Metric": [
            "Total features",
            "Drifted features",
            "Drift share",
            "Dataset drift threshold",
            "Overall drift detected"
        ],
        "Value": [
            total_features,
            drifted_count,
            f"{drift_share:.1%}",
            f"{drift_threshold:.1%}",
            drift_share >= drift_threshold
        ]
    })

def get_drift_share(drift_result):
    """Extract overall drift share from Evidently result."""

    for metric in drift_result["metrics"]:
        if metric["config"]["type"] == "evidently:metric_v2:DriftedColumnsCount":
            return {
                "drifted_features": int(metric["value"]["count"]),
                "drift_share": metric["value"]["share"]
            }

    raise ValueError("DriftedColumnsCount not found")

def historical_drift_backtest(
    X,
    start_year=2021,
    max_round=14,
    exclude_cols=None
):

    if exclude_cols is None:
        exclude_cols = []

    results = []

    years = sorted(X["Season"].unique())

    for year in years:

        if year < start_year:
            continue

        # Everything BEFORE this season = historical reference
        X_reference = X[X["Season"] < year].copy()

        # Pretend this season is the "new" data
        # Only use the first N rounds, matching your current 2026 situation
        X_current = X[
            (X["Season"] == year) &
            (X["race_round"] <= max_round)
        ].copy()

        if len(X_reference) == 0 or len(X_current) == 0:
            continue

        # Don't allow purely temporal / ID variables
        # to dominate the drift decision
        reference_drift = X_reference.drop(
            columns=exclude_cols,
            errors="ignore"
        )

        current_drift = X_current.drop(
            columns=exclude_cols,
            errors="ignore"
        )

        drift_result = check_data_drift(
            X_old=reference_drift,
            X_new=current_drift
        )

        summary = get_drift_share(drift_result)

        results.append({
            "year": year,
            "reference_years":
                f"{int(X_reference['Season'].min())}-{year-1}",
            "current_rounds": f"1-{max_round}",
            "n_reference": len(X_reference),
            "n_current": len(X_current),
            "drifted_features": summary["drifted_features"],
            "drift_share": summary["drift_share"]
        })

    return pd.DataFrame(results)

def historical_drift_backtest_transformed(
    X_raw,
    fitted_pipeline,
    start_year=2021,
    max_round=14
):

    # ---------------------------------
    # Transform using FITTED pipeline
    # ---------------------------------
    step_names = list(fitted_pipeline.named_steps.keys())
    selector_idx = step_names.index("Feature_Selector")

    # Everything before selector => here we perform 
    X_encoded = fitted_pipeline[:selector_idx].transform(X_raw)

    # Apply fitted selector
    selector = fitted_pipeline["Feature_Selector"]
    X_selected = selector.transform(X_encoded)

    encoded_cols = X_selected.columns.difference(X_raw.columns,sort=False)
    X_selected = X_selected[encoded_cols].copy()

    results = []

    years = sorted(X_raw["Season"].unique())

    for year in years:

        if year < start_year:
            continue

        reference_idx = X_raw.index[
            X_raw["Season"] < year
        ]

        current_idx = X_raw.index[
            (X_raw["Season"] == year) &
            (X_raw["race_round"] <= max_round)
        ]

        if len(reference_idx) == 0 or len(current_idx) == 0:
            continue

        X_reference = X_selected.loc[reference_idx]
        X_current = X_selected.loc[current_idx]

        drift_result = check_data_drift(
            X_old=X_reference,
            X_new=X_current,
            drift_share=0.5
        )

        summary = get_drift_share(drift_result)

        results.append({
            "year": year,
            "reference_years":
                f"{int(X_raw.loc[reference_idx, 'Season'].min())}-{year-1}",
            "current_rounds": f"1-{max_round}",
            "n_reference": len(X_reference),
            "n_current": len(X_current),
            "drifted_features": summary["drifted_features"],
            "drift_share": summary["drift_share"]
        })

    return pd.DataFrame(results)

## Raw Data Drift Detection

In [11]:
drift_result = check_data_drift(X_old=X_old,X_new=X_new)
drift_table = make_drift_table(drift_result)

drift_table

,feature,method,score,threshold,drift_detected
0,AirTemp_p1,Wasserstein distance (normed),2.890976e-01,0.10,True
1,Humidity_p1,Wasserstein distance (normed),6.230890e-01,0.10,True
2,Pressure_p1,Wasserstein distance (normed),3.152400e-01,0.10,True
3,laptime_sum_sectortimes_p1,Wasserstein distance (normed),1.582607e-01,0.10,True
4,LapTimeDiff_p1,Wasserstein distance (normed),1.784493e-01,0.10,True
5,AirTemp_p2,Wasserstein distance (normed),3.032634e-01,0.10,True
6,Humidity_p2,Wasserstein distance (normed),5.376682e-01,0.10,True
7,Pressure_p2,Wasserstein distance (normed),2.647875e-01,0.10,True
8,laptime_sum_sectortimes_p2,Wasserstein distance (normed),3.272193e-01,0.10,True
9,LapTimeDiff_p2,Wasserstein distance (normed),2.206322e-01,0.10,True


In [12]:
get_drift_summary(drift_result)

,Metric,Value
0,Total features,55
1,Drifted features,48
2,Drift share,87.3%
3,Dataset drift threshold,92.0%
4,Overall drift detected,False


In [13]:
exclude_cols = (
    [c for c in X_old.columns if "sprint" in c.lower()]
    + ["Season", "gp_id"]
)

historical_drift = historical_drift_backtest(
    X=X_old,
    start_year=2021,
    max_round=11,
    exclude_cols=exclude_cols
)

historical_drift

,year,reference_years,current_rounds,n_reference,n_current,drifted_features,drift_share
0,2021,2018-2020,1-11,1111,214,38,0.883721
1,2022,2018-2021,1-11,1544,216,40,0.930233
2,2023,2018-2022,1-11,1980,219,37,0.860465
3,2024,2018-2023,1-11,2418,218,40,0.930233
4,2025,2018-2024,1-11,2895,218,41,0.953488


In [14]:
features = [
    c for c in X_old.columns
    if c not in exclude_cols
]

result_2026 = check_data_drift(
    X_old=X_old[features],
    X_new=X_new[features]
)

summary_2026 = get_drift_share(result_2026)

print(summary_2026)

{'drifted_features': 39, 'drift_share': 0.9069767441860465}


In [15]:
X_reference_2026 = X_old.drop(
    columns=["Season", "gp_id"],
    errors="ignore"
)

X_current_2026 = X_new.drop(
    columns=["Season", "gp_id"],
    errors="ignore"
)

result_2026 = check_data_drift(
    X_old=X_reference_2026,
    X_new=X_current_2026
)

summary_2026 = get_drift_share(result_2026)

print(summary_2026)

{'drifted_features': 46, 'drift_share': 0.8679245283018868}


In [16]:
historical_drift["drift_share"].describe()

count    5.000000
mean     0.911628
std      0.038213
min      0.860465
25%      0.883721
50%      0.930233
75%      0.930233
max      0.953488
Name: drift_share, dtype: float64

In [17]:
historical_drift["drift_share"].quantile([
    0.50,
    0.75,
    0.90,
    0.95
])

0.50    0.930233
0.75    0.930233
0.90    0.944186
0.95    0.948837
Name: drift_share, dtype: float64

In [18]:
dd_v = historical_drift["drift_share"].describe()
dd_v['max']

0.9534883720930233

In [19]:
drift_result = check_data_drift(X_old=X_old,X_new=X_new,drift_share=dd_v['max'].round(2))
drift_table = make_drift_table(drift_result)

drift_table

,feature,method,score,threshold,drift_detected
0,AirTemp_p1,Wasserstein distance (normed),2.890976e-01,0.10,True
1,Humidity_p1,Wasserstein distance (normed),6.230890e-01,0.10,True
2,Pressure_p1,Wasserstein distance (normed),3.152400e-01,0.10,True
3,laptime_sum_sectortimes_p1,Wasserstein distance (normed),1.582607e-01,0.10,True
4,LapTimeDiff_p1,Wasserstein distance (normed),1.784493e-01,0.10,True
5,AirTemp_p2,Wasserstein distance (normed),3.032634e-01,0.10,True
6,Humidity_p2,Wasserstein distance (normed),5.376682e-01,0.10,True
7,Pressure_p2,Wasserstein distance (normed),2.647875e-01,0.10,True
8,laptime_sum_sectortimes_p2,Wasserstein distance (normed),3.272193e-01,0.10,True
9,LapTimeDiff_p2,Wasserstein distance (normed),2.206322e-01,0.10,True


In [20]:
get_drift_summary(drift_result)

,Metric,Value
0,Total features,55
1,Drifted features,48
2,Drift share,87.3%
3,Dataset drift threshold,95.0%
4,Overall drift detected,False


## Selected Feature & Encoded Features

In [ ]:
current = Path.cwd()

PROJECT_ROOT = None

for folder in [current, *current.parents]:
    if (folder / "src" / "py_def_class.py").exists():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

# Make `src` importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Optional sanity check
import src.py_def_class

fm = joblib.load(
    PROJECT_ROOT
    / "model"
    / "final_model"
    / "final_model_qualifying.joblib"
)

c:\Users\gandj\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning:

Trying to unpickle estimator DecisionTreeRegressor from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations

c:\Users\gandj\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning:

Trying to unpickle estimator RandomForestRegressor from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations

c:\Users\gandj\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning:

Trying to unpickle estimator Pipeline from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid resu

In [22]:
fm.named_steps.keys()

dict_keys(['RareLabel_Driver', 'RareLabelEncoder_GP', 'RareLabelEncoder_Compounds', 'Adding_Columns_for_mean_&_frequency', 'Indicating_Missing_Value', 'Drop_Features', 'One_Hot_Encoded_Features', 'Mean_encoding', 'Frequency_encoding', 'Feature_Selector', 'Light_GBM'])

In [23]:
step_names = list(fm.named_steps.keys())
selector_idx = step_names.index("Feature_Selector")

# Apply everything BEFORE the feature selector
X_old_encoded = fm[:selector_idx].transform(X_old)
X_new_encoded = fm[:selector_idx].transform(X_new)

# Now apply the feature selector
selector = fm["Feature_Selector"]
X_old_selected = selector.transform(X_old_encoded)
X_new_selected = selector.transform(X_new_encoded)

encoded_cols = set(X_old_selected.columns)
raw_cols = set(X_old.columns)

remaining_cols = encoded_cols - raw_cols
remaining_cols = [c for c in remaining_cols]

X_old_enc_select = X_old_selected[remaining_cols]
X_new_enc_select = X_new_selected[remaining_cols]

Here we have filtered out the columns that have been already test in the raw data drift section.

In [24]:
#here we run all featured
historical_encoded_drift = historical_drift_backtest_transformed(
    X_raw=X_old,
    fitted_pipeline=fm,
    start_year=2021,
    max_round=11
)

historical_encoded_drift

,year,reference_years,current_rounds,n_reference,n_current,drifted_features,drift_share
0,2021,2018-2020,1-11,1111,214,2,1.0
1,2022,2018-2021,1-11,1544,216,2,1.0
2,2023,2018-2022,1-11,1980,219,2,1.0
3,2024,2018-2023,1-11,2418,218,2,1.0
4,2025,2018-2024,1-11,2895,218,2,1.0


Based on that outcome, we will use our default drift_sahre of 0.75, because as we can see, we mainly have a drift of 100%.

In [25]:
drift_result = check_data_drift(X_old=X_old_enc_select,X_new=X_new_enc_select,drift_share=0.5)
drift_table = make_drift_table(drift_result)

drift_table

,feature,method,score,threshold,drift_detected
0,Compound_quali_mean,Wasserstein distance (normed),0.331257,0.1,True
1,GP_mean,Wasserstein distance (normed),0.190519,0.1,True


In [26]:
get_drift_summary(drift_result)

,Metric,Value
0,Total features,2
1,Drifted features,2
2,Drift share,100.0%
3,Dataset drift threshold,50.0%
4,Overall drift detected,True


Next we will analyse the target drift.

## Target Drift

In [27]:
y_old = pd.read_csv(r'../data/data_drift/target_feature.csv')
y_new = pd.read_csv(r'../data/data_drift/y_new.csv')

In [5]:
def historical_drift_backtest_target(
    X_data_for_year,
    y,
    start_year=2021,
    max_round=14,
    exclude_cols=None
):

    if exclude_cols is None:
        exclude_cols = []

    results = []

    years = sorted(X_data_for_year["Season"].unique())

    for year in years:

        if year < start_year:
            continue

        # Everything BEFORE this season = historical reference
        X_reference = X_data_for_year[X_data_for_year["Season"] < year].copy()

        # Pretend this season is the "new" data
        # Only use the first N rounds, matching your current 2026 situation
        X_current = X_data_for_year[
            (X_data_for_year["Season"] == year) &
            (X_data_for_year["race_round"] <= max_round)
        ].copy()

        if len(X_reference) == 0 or len(X_current) == 0:
            continue

        year_index_ref = X_reference.index
        year_index_current = X_current.index
        

        # Don't allow purely temporal / ID variables
        # to dominate the drift decision
        reference_drift = y.loc[year_index_ref].to_frame(name="target")

        current_drift = y.loc[year_index_current].to_frame(name="target")

        drift_result = check_data_drift(
            X_old=reference_drift,
            X_new=current_drift
        )

        summary = get_drift_share(drift_result)

        results.append({
            "year": year,
            "reference_years":
                f"{int(X_reference['Season'].min())}-{year-1}",
            "current_rounds": f"1-{max_round}",
            "n_reference": len(X_reference),
            "n_current": len(X_current),
            "drifted_features": summary["drifted_features"],
            "drift_share": summary["drift_share"]
        })

    return pd.DataFrame(results)

In [28]:
historical_drift_backtest_target(
    X_data_for_year=X_old,
    y=y_old,
    start_year=2021,
    max_round=11,
    exclude_cols=None
)

,year,reference_years,current_rounds,n_reference,n_current,drifted_features,drift_share
0,2021,2018-2020,1-11,1111,214,1,1.0
1,2022,2018-2021,1-11,1544,216,1,1.0
2,2023,2018-2022,1-11,1980,219,1,1.0
3,2024,2018-2023,1-11,2418,218,1,1.0
4,2025,2018-2024,1-11,2895,218,1,1.0


In [ ]:
drift_result = check_data_drift(X_old=y_old,X_new=y_new,drift_share=0.5)
drift_table = make_drift_table(drift_result)
drift_table

,feature,method,score,threshold,drift_detected
0,laptime_sum_sectortimes_quali,Wasserstein distance (normed),0.158593,0.1,True


## Comparing the metrics

In [33]:
mae_old = joblib.load(r'..\data\data_drift\final_model_mae_test_performance.joblib')
rsme_old = joblib.load(r'..\data\data_drift\final_model_rsme_test_performance.joblib')
mape_old = joblib.load(r'..\data\data_drift\final_model_mape_test_performance.joblib')

new_pred = fm.predict(X_new)
mae_new = mean_absolute_error(y_new,new_pred)
rmse_new = root_mean_squared_error(y_new,new_pred)
mape_new = mean_absolute_percentage_error(y_new,new_pred)

diff_mae = mae_new - mae_old
print(f'\n{"Model got significantly worse" if diff_mae > 0.150 else "Current model leads to reliable results"}\n')

print(f'Old MAE: {mae_old} - New MAE: {mae_new}')
print(f'Old RSME: {rsme_old} - New RSME: {rmse_new}')
print(f'Old MAPE: {mape_old} - New RSME: {mape_new}')

[LightGBM] [Warning] lambda_l1 is set with reg_alpha=0.0, l1_regularization=1.7377001319324172 will be ignored. Current value: lambda_l1=0.0
[LightGBM] [Warning] lambda_l2 is set with reg_lambda=0.0, l2_regularization=1.7842059975472844 will be ignored. Current value: lambda_l2=0.0
[LightGBM] [Warning] min_data_in_leaf is set=14, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=14
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.06590921163569698, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.06590921163569698
[LightGBM] [Warning] bagging_fraction is set=0.5906195061033961, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5906195061033961
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3

Current model leads to reliable results

Old MAE: 1.0750876063598818 - New MAE: 0.7923169936872931
Old RSME: 2.429885284408754 - New RSME: 1.1754967459257457
Old MAPE: